# 04 — Explainability with SHAP
Why does the model flag a given customer as high-risk?

In [ ]:
import pandas as pd
import joblib
import shap
import glob
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/churn_processed.csv')
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model_path = glob.glob('../models/best_model_*.pkl')[0]
model = joblib.load(model_path)
print('Loaded:', model_path)

## Global feature importance — SHAP summary plot

In [ ]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, show=False)
import matplotlib.pyplot as plt
plt.savefig('../reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Explaining individual predictions
Pick a couple of high-risk customers and show exactly which features pushed their churn probability up.

In [ ]:
probs = model.predict_proba(X_test)[:, 1]
top_risk_idx = probs.argsort()[::-1][:3]

for i in top_risk_idx:
    print(f'Customer index {i} — predicted churn probability: {probs[i]:.2f}')
    shap.plots.waterfall(shap_values[i], show=True)

## Takeaways for the business (fill in after running)
- Which features push risk up the most, on average?
- Are there any surprising interactions (e.g. a feature that matters a lot only for certain customers)?
- What would you recommend a retention team prioritize based on this?